In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
BASE_DATASET = r"C:\Users\manoj\Downloads\Network_Datasets"
PROCESSED_PATH = r"C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\data\processed"
MODEL_PATH = r"C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\models"

In [3]:
scaled_df = pd.read_parquet(os.path.join(PROCESSED_PATH, "final_scaled_dataset.parquet"))
oof_probs = pd.read_parquet(os.path.join(MODEL_PATH, "stage1_oof_probs.parquet"))

stage2_base = pd.concat([scaled_df, oof_probs], axis=1)

print(stage2_base.shape)

(1650000, 19)


In [4]:
def map_attack_category(dataset, value):

    value = str(value).lower()

    # ---------- TON ----------
    if dataset == "TON":

        if value in ["normal"]:
            return "Normal"

        elif "dos" in value or "ddos" in value:
            return "DoS/DDoS"

        elif "scan" in value:
            return "Probe"

        elif "backdoor" in value:
            return "Infiltration"

        elif "ransomware" in value:
            return "Exploit"

        elif "bot" in value:
            return "Botnet"

        else:
            return "Exploit"

    # ---------- CIC ----------
    elif dataset == "CIC":

        if "benign" in value:
            return "Normal"

        elif "dos" in value or "ddos" in value:
            return "DoS/DDoS"

        elif "portscan" in value:
            return "Probe"

        elif "web" in value or "sql" in value or "xss" in value:
            return "Web"

        elif "brute" in value or "ftp" in value or "ssh" in value:
            return "Brute Force"

        elif "infiltration" in value:
            return "Infiltration"

        else:
            return "Exploit"

    # ---------- UNSW ----------
    elif dataset == "UNSW":

        if value == "normal":
            return "Normal"

        elif value in ["dos", "ddos"]:
            return "DoS/DDoS"

        elif value in ["reconnaissance"]:
            return "Probe"

        elif value in ["exploits", "fuzzers", "generic"]:
            return "Exploit"

        elif value in ["backdoor"]:
            return "Infiltration"

        elif value in ["worms"]:
            return "Botnet"

        else:
            return "Exploit"

    return "Unknown"

In [5]:
ton_folder = os.path.join(BASE_DATASET, "TON IOT Dataset")

ton_labels = []

for file in tqdm(os.listdir(ton_folder)):
    path = os.path.join(ton_folder, file)
    
    df = pd.read_csv(path, usecols=["label", "type"], low_memory=False)
    df = df.sample(min(50000, len(df)), random_state=42)
    
    df["attack_type"] = df["type"].apply(
        lambda x: map_attack_category("TON", x)
    )
    
    ton_labels.append(df[["attack_type"]])

ton_labels = pd.concat(ton_labels, ignore_index=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 23/23 [01:19<00:00,  3.45s/it]


In [6]:
cic_folder = os.path.join(BASE_DATASET, "CIC IDS Dataset")

cic_labels = []

for file in tqdm(os.listdir(cic_folder)):
    path = os.path.join(cic_folder, file)
    
    df = pd.read_csv(path, usecols=[" Label"], low_memory=False)
    df = df.sample(min(50000, len(df)), random_state=42)
    
    df["attack_type"] = df[" Label"].apply(
        lambda x: map_attack_category("CIC", x)
    )
    
    cic_labels.append(df[["attack_type"]])

cic_labels = pd.concat(cic_labels, ignore_index=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:19<00:00,  2.39s/it]


In [7]:
unsw_train = pd.read_csv(os.path.join(BASE_DATASET, "UNSW Dataset", "UNSW_NB15_training-set.csv"))
unsw_test = pd.read_csv(os.path.join(BASE_DATASET, "UNSW Dataset", "UNSW_NB15_testing-set.csv"))

unsw_df = pd.concat([unsw_train, unsw_test], ignore_index=True)
unsw_df = unsw_df.sample(100000, random_state=42)

unsw_df["attack_type"] = unsw_df["attack_cat"].apply(
    lambda x: map_attack_category("UNSW", x)
)

unsw_labels = unsw_df[["attack_type"]]

In [8]:
all_labels = pd.concat([ton_labels, cic_labels, unsw_labels], ignore_index=True)

print("Labels shape:", all_labels.shape)

Labels shape: (1650000, 1)


In [9]:
print(stage2_base.shape)
print(all_labels.shape)

(1650000, 19)
(1650000, 1)


In [10]:
stage2_full = pd.concat([stage2_base.reset_index(drop=True), all_labels], axis=1)

print(stage2_full.shape)

(1650000, 20)


In [11]:
stage2_full = stage2_full[stage2_full["attack_type"] != "Normal"].reset_index(drop=True)

print(stage2_full["attack_type"].value_counts())

attack_type
DoS/DDoS        529766
Probe           390173
Exploit         269585
Infiltration     56065
Brute Force       1560
Web                652
Botnet              65
Name: count, dtype: int64


In [12]:
label_mapping = {
    "DoS/DDoS": 0,
    "Probe": 1,
    "Web": 2,
    "Brute Force": 3,
    "Botnet": 4,
    "Exploit": 5,
    "Infiltration": 6
}

stage2_full["attack_label"] = stage2_full["attack_type"].map(label_mapping)

In [13]:
features = stage2_full.drop(["label", "attack_type", "attack_label"], axis=1)
labels = stage2_full["attack_label"]

print(features.shape, labels.shape)

(1247866, 18) (1247866,)


In [14]:
output_file = os.path.join(PROCESSED_PATH, "stage2_dataset.parquet")

stage2_full.to_parquet(output_file, index=False)

print("Stage-2 dataset saved:", output_file)

Stage-2 dataset saved: C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\data\processed\stage2_dataset.parquet
